# Chronos-2 Last Latent Demo

This notebook shows how to call `pipeline.predict_last_latent(...)` with the same input formats accepted by `pipeline.predict_quantiles(...)`.

The returned latent is the final encoder layer's last future-patch representation: `encoder_outputs.last_hidden_state[:, -1, :]`.

## 1. Load Chronos-2

Use CPU for portability. Switch `device_map="cuda"` if a GPU is available.

In [8]:
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Force this notebook to import Chronos from the local repository checkout,
# not from a previously installed `chronos-forecasting` package.
cwd = Path.cwd().resolve()
repo_root = next(path for path in [cwd, *cwd.parents] if (path / "src" / "chronos").exists())
local_src = str(repo_root / "src")
if local_src not in sys.path:
    sys.path.insert(0, local_src)

# If the kernel already imported chronos from site-packages, clear it so the
# next import resolves from local_src above.
for module_name in list(sys.modules):
    if module_name == "chronos" or module_name.startswith("chronos."):
        del sys.modules[module_name]

import numpy as np
import torch

from chronos import BaseChronosPipeline, Chronos2Pipeline

print("Using local Chronos package from:", Path(sys.modules["chronos"].__file__).resolve())

pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
)

prediction_length = 24
assert hasattr(pipeline, "predict_last_latent")

Using local Chronos package from: /Users/lige/code/TimeSeriesForecast/chronos-forecasting/src/chronos/__init__.py


## 2. Univariate Batch

`predict_last_latent` accepts the same 3D tensor input as `predict_quantiles`: `(batch, n_variates, history_length)`.

In [9]:
rng = np.random.default_rng(seed=0)
inputs = rng.normal(size=(8, 2, 128)).astype(np.float32)

quantiles, median = pipeline.predict_quantiles(
    inputs,
    prediction_length=prediction_length,
    quantile_levels=[0.1, 0.5, 0.9],
)

last_latents = pipeline.predict_last_latent(
    inputs,
    prediction_length=prediction_length,
)

print("Number of forecast items:", len(quantiles))
print("Quantiles[0] shape:", quantiles[0].shape)  # (n_variates, prediction_length, n_quantiles)
print("Median[0] shape:", median[0].shape)        # (n_variates, prediction_length)
print("Latent[0] shape:", last_latents[0].shape)  # (n_variates, d_model)

Number of forecast items: 8
Quantiles[0] shape: torch.Size([2, 24, 3])
Median[0] shape: torch.Size([2, 24])
Latent[0] shape: torch.Size([2, 768])


The latent list is aligned with the forecast list. For a univariate item, each latent has shape `(1, d_model)`.

In [10]:
representation_matrix = torch.cat(last_latents, dim=0)

print("Representation matrix shape:", representation_matrix.shape)  # (batch, d_model)
print("Representation dtype:", representation_matrix.dtype)

Representation matrix shape: torch.Size([16, 768])
Representation dtype: torch.float32


## 3. Multivariate Batch

For multivariate input, Chronos-2 groups variates from the same item internally. The returned latent keeps that structure: `(n_variates, d_model)` per input item.

In [11]:
multivariate_inputs = rng.normal(size=(4, 3, 256)).astype(np.float32)

multi_latents = pipeline.predict_last_latent(
    multivariate_inputs,
    prediction_length=prediction_length,
)

print("Number of multivariate forecast items:", len(multi_latents))
print("Latent[0] shape:", multi_latents[0].shape)  # (3, d_model)

Number of multivariate forecast items: 4
Latent[0] shape: torch.Size([3, 768])


## 4. Inputs With Covariates

Dictionary inputs also work. The latent output is returned only for target variates, matching the forecast outputs.

In [12]:
covariate_inputs = [
    {
        "target": rng.normal(size=160).astype(np.float32),
        "past_covariates": {
            "temperature": rng.normal(size=160).astype(np.float32),
            "promotion": rng.integers(0, 2, size=160).astype(np.float32),
        },
        "future_covariates": {
            "promotion": rng.integers(0, 2, size=prediction_length).astype(np.float32),
        },
    }
    for _ in range(3)
]

cov_quantiles, cov_median = pipeline.predict_quantiles(
    covariate_inputs,
    prediction_length=prediction_length,
    quantile_levels=[0.1, 0.5, 0.9],
)
cov_latents = pipeline.predict_last_latent(
    covariate_inputs,
    prediction_length=prediction_length,
)

print("Quantiles[0] shape:", cov_quantiles[0].shape)
print("Latent[0] shape:", cov_latents[0].shape)

Quantiles[0] shape: torch.Size([1, 24, 3])
Latent[0] shape: torch.Size([1, 768])


## Notes

- `predict_last_latent(...)` does not change `predict_quantiles(...)`; it reuses the same preprocessing path and calls `model.encode(...)` directly.
- The latent corresponds to the last generated future patch in the first forward pass. If `prediction_length` is longer than the model's direct horizon, this method still returns the representation from the first direct encode call rather than from later autoregressive unrolls.
- Returned tensors are moved to CPU with `float32` dtype so they can be used directly for downstream representation learning, clustering, retrieval, or diagnostics.